# 02. Exploratory Data Analysis (EDA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week12/02.Exploratory-Data-Analysis/notebooks/01_02.Exploratory-Data-Analysis.ipynb)

## Learning Objectives
- Apply a formal 5-step EDA framework (Structure -> Distribution & Outliers -> Categorical Breakdown -> Bivariate Relationships -> Synthesis).
- Inspect data shapes, missing entries, and data types defensively.
- Detect numerical anomalies and extreme values using Tukey's Fence ($1.5 \times \text{IQR}$ rule) and boxplots.
- Compute and interpret correlation matrices between academic metrics and compensation.
- Group and segment outcomes to uncover hidden trends across demographic categories.


## 1. Step 1: Structural & Shape Audit
Before performing any calculations or visualisations, we inspect dataset dimensions, data types, and check for unexpected missing or corrupted records.
Let's load the Australian Graduate Outcomes Survey dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)
n = 120

study_areas = np.random.choice(
    ['Software Engineering', 'Data Science', 'Cyber Security', 'Business Analytics'],
    size=n, p=[0.35, 0.25, 0.20, 0.20]
)
internships = np.random.choice(['Yes', 'No'], size=n, p=[0.60, 0.40])
gpa = np.random.normal(loc=5.5, scale=0.8, size=n).clip(3.5, 7.0).round(2)

discipline_boost = {
    'Software Engineering': 72000,
    'Data Science': 75000,
    'Cyber Security': 74000,
    'Business Analytics': 68000
}
base = np.array([discipline_boost[d] for d in study_areas])
internship_bonus = np.where(internships == 'Yes', 6500, 0)
salary = base + internship_bonus + (gpa - 5.0) * 4000 + np.random.normal(0, 4500, n)
salary = salary.round(-2)

# Inject realistic outliers: 1 executive contract outlier, 1 typo outlier
salary[15] = 240000.0
salary[42] = 7200.0

satisfaction = np.random.randint(55, 98, size=n)

df = pd.DataFrame({
    'graduate_id': [f'GRAD-{2026000 + i}' for i in range(n)],
    'study_area': study_areas,
    'internship': internships,
    'gpa': gpa,
    'starting_salary': salary,
    'satisfaction_score': satisfaction
})

print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
display(df.head())
print('\nData Types & Info:')
print(df.dtypes)

## 2. Step 2: Distribution Analysis & Outlier Detection
Summary statistics (mean, standard deviation) can be severely distorted by extreme values.
We compute the Five-Number Summary (Min, Q1, Median, Q3, Max) and apply Tukey's Rule:
- $\text{IQR} = Q_3 - Q_1$
- $\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}$
- $\text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$

In [ ]:
summary = df[['gpa', 'starting_salary', 'satisfaction_score']].describe().T
display(summary[['mean', 'std', 'min', '25%', '50%', '75%', 'max']])

# Tukey's Fence for Starting Salary
q1 = df['starting_salary'].quantile(0.25)
q3 = df['starting_salary'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f'Salary Q1: ${q1:,.2f} | Q3: ${q3:,.2f} | IQR: ${iqr:,.2f}')
print(f'Lower Bound: ${lower_bound:,.2f} | Upper Bound: ${upper_bound:,.2f}')

outliers = df[(df['starting_salary'] < lower_bound) | (df['starting_salary'] > upper_bound)]
print(f'\nDetected {len(outliers)} Outlier(s):')
display(outliers[['graduate_id', 'study_area', 'starting_salary', 'internship']])

## 3. Step 3: Categorical Breakdown & Proportions
We inspect the distribution of graduates across study areas and internship status using `.value_counts()`.

In [ ]:
counts = df['study_area'].value_counts()
proportions = df['study_area'].value_counts(normalize=True).mul(100).round(1)
cat_df = pd.DataFrame({'Count': counts, 'Percentage (%)': proportions})
display(cat_df)

print('Internship Completion Rate (%):')
display(df['internship'].value_counts(normalize=True).mul(100).round(1))

## 4. Step 4: Bivariate Correlation & Group Analysis
How does academic achievement (GPA) correlate with graduate compensation? And how does an industry internship impact median salary across disciplines?

In [ ]:
# Pearson Correlation Matrix
corr = df[['gpa', 'starting_salary', 'satisfaction_score']].corr().round(3)
print('Correlation Matrix:')
display(corr)

# Segmented Group Summary: Median & Mean Salary by Discipline and Internship
grouped = df.groupby(['study_area', 'internship'])['starting_salary'].agg(['count', 'median', 'mean']).round(0)
print('\nCompensation by Study Area & Internship Experience:')
display(grouped)

## 5. Visualising EDA Insights
Let's create side-by-side visualisations: a boxplot highlighting the outlier salaries across disciplines, and a scatter plot illustrating the relationship between GPA, internships, and starting salary.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
study_order = sorted(df['study_area'].unique())
salary_data = [df[df['study_area'] == area]['starting_salary'] for area in study_order]
axes[0].boxplot(salary_data, tick_labels=study_order, patch_artist=True)
axes[0].set_title('Graduate Starting Salary by Discipline (Boxplot)', fontweight='bold')
axes[0].set_ylabel('Starting Salary (AUD)')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, linestyle='--', alpha=0.5, axis='y')

# Scatter Plot
intern_yes = df[df['internship'] == 'Yes']
intern_no = df[df['internship'] == 'No']
axes[1].scatter(intern_yes['gpa'], intern_yes['starting_salary'], color='#2b5c8f', alpha=0.7, label='Internship Completed')
axes[1].scatter(intern_no['gpa'], intern_no['starting_salary'], color='#d95f02', alpha=0.7, label='No Internship')
axes[1].axhline(upper_bound, color='red', linestyle=':', label=f'Outlier Threshold (${upper_bound:,.0f})')
axes[1].set_title('Starting Salary vs. Academic GPA', fontweight='bold')
axes[1].set_xlabel('GPA (3.5 - 7.0 Scale)')
axes[1].set_ylabel('Starting Salary (AUD)')
axes[1].legend(loc='upper left')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 6. Practice Exercises

### Exercise: Detecting Score Anomalies
Given a student quiz results DataFrame, compute the IQR for `exam_score` and identify any students whose scores fall beyond the lower or upper fence.

In [ ]:
# Exercise Data
quiz_data = pd.DataFrame({
    'student_id': [f'S{i}' for i in range(1, 9)],
    'exam_score': [65, 72, 88, 45, 92, 78, 85, 15],
    'study_hours': [12, 15, 22, 8, 25, 18, 20, 3]
})
display(quiz_data)

# --- Student Solution ---
q1 = quiz_data['exam_score'].quantile(0.25)
q3 = quiz_data['exam_score'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = quiz_data[(quiz_data['exam_score'] < lower) | (quiz_data['exam_score'] > upper)]
print(f'IQR: {iqr:.1f} | Lower Fence: {lower:.1f} | Upper Fence: {upper:.1f}')
print('Identified Outlier(s):')
display(outliers)